In [ ]:
import pandas as pd
import numpy as np
import os
import random
import librosa
import matplotlib.pyplot as plt

In [ ]:
new_label = pd.read_csv('../audio_data/8k/labels.csv')
old_label = pd.read_csv('../audio_data/2k/labels.csv')

In [ ]:
old_label

In [ ]:
new_label['filename'] = new_label['fold'].astype('str') + '-' + new_label['slice_file_name']

In [ ]:
new_label.to_csv('../audio_data/8k/labels.csv', index=False)

In [ ]:
# To get a list of the pathes of all the audio files
data = "../audio_data/8k/22050/"
dataset = pd.read_csv('../audio_data/8k/labels.csv')
all_files = []
for path, subdirs, files in os.walk(data):
    for name in files:
        all_files.append(os.path.join(path, name))
        
# A dictionary to decode the categories into targets
#decoder = {0: 'dog', 14: 'chirping_birds', 36: 'vacuum_cleaner', 19: 'thunderstorm', 30: 'door_wood_knock',34: 'can_opening', 9: 'crow', 22: 'clapping', 48: 'fireworks', 41: 'chainsaw', 47: 'airplane', 31: 'mouse_click', 17: 'pouring_water', 45: 'train', 8: 'sheep', 15: 'water_drops', 46: 'church_bells', 37: 'clock_alarm', 32: 'keyboard_typing', 16: 'wind', 25: 'footsteps', 4: 'frog', 3: 'cow', 27: 'brushing_teeth', 43: 'car_horn', 12: 'crackling_fire', 40: 'helicopter', 29: 'drinking_sipping', 10: 'rain', 7: 'insects', 26: 'laughing', 6: 'hen', 44: 'engine', 23: 'breathing', 20: 'crying_baby', 49: 'hand_saw', 24: 'coughing', 39: 'glass_breaking', 28: 'snoring', 18: 'toilet_flush', 2: 'pig', 35: 'washing_machine', 38: 'clock_tick', 21: 'sneezing', 1: 'rooster', 11: 'sea_waves', 42: 'siren', 5: 'cat', 33: 'door_wood_creaks', 13: 'crickets'}

# A dictionary to encode the categories into targets
#encoder = {'dog': 0, 'chirping_birds': 14, 'vacuum_cleaner': 36, 'thunderstorm': 19, 'door_wood_knock': 30, 'can_opening': 34, 'crow': 9, 'clapping': 22, 'fireworks': 48, 'chainsaw': 41, 'airplane': 47, 'mouse_click': 31, 'pouring_water': 17, 'train': 45, 'sheep': 8, 'water_drops': 15, 'church_bells': 46, 'clock_alarm': 37, 'keyboard_typing': 32, 'wind': 16, 'footsteps': 25, 'frog': 4, 'cow': 3, 'brushing_teeth': 27, 'car_horn': 43, 'crackling_fire': 12, 'helicopter': 40, 'drinking_sipping': 29, 'rain': 10, 'insects': 7, 'laughing': 26, 'hen': 6, 'engine': 44, 'breathing': 23, 'crying_baby': 20, 'hand_saw': 49, 'coughing': 24, 'glass_breaking': 39, 'snoring': 28, 'toilet_flush': 18, 'pig': 2, 'washing_machine': 35, 'clock_tick': 38, 'sneezing': 21, 'rooster': 1, 'sea_waves': 11, 'siren': 42, 'cat': 5, 'door_wood_creaks': 33, 'crickets': 13}

In [ ]:
sorted_files = sorted(all_files)
sorted_files[-4:]

In [ ]:
new_label.sort_values(by=['filename'], inplace=True)
new_label.reset_index(drop=True, inplace=True)
new_label

In [ ]:
sample_rate_ls = []
duration_ls = []

for file in all_files:
    y, sr = librosa.load(file)
    sample_rate_ls.append(sr)
    duration_ls.append(librosa.get_duration(y=y, sr=sr))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(duration_ls)

In [ ]:
new_label['duration'] = duration_ls
new_label.sort_values(by='duration', ascending=False)

In [ ]:
new_label['check'] = new_label.duration.apply(lambda x: 1 if x >= 4.0 else 0)

In [ ]:
new_label.check.value_counts()

In [ ]:
plot_files = random.choices(all_files, k = 10)
plot_audios = [librosa.load(plot_files[i]) for i in range(10)]

In [ ]:
# Importing 1 file
y, sr = librosa.load(data + "8-61077-3-1-0.wav")

print('y:', y, '\n')
print('y shape:', np.shape(y), '\n')
print('Sample Rate (KHz):', sr, '\n')

# The duration is equal to the number of frames divided by the framerate
print('Duration of the audio file:', np.shape(y)[0]/sr, 'second')

In [ ]:
y

## 1. Audio-waves

In [ ]:
plt.figure(figsize=(30,30))
for i in range(1,10):
    plt.subplot(3,3,i)
    librosa.display.waveshow(plot_audios[i][0])
    try:
        plt.title("Sound of " + decoder[int(plot_files[i][-6:-4])] )
    except:
        plt.title("Sound of " + decoder[int(plot_files[i][-5:-4])] )

## 2. STFT

- time-domain signal to frequency-domain representation of the signal
- relative amplitudes of the different frequency components that make up the signal
- breaks the audio signal into short segments and applies the DFT to each segment

In [ ]:
# Default FFT window size
n_fft = 2048 # FFT window size
hop_length = 512 # number audio of frames between STFT columns 

plt.figure(figsize=(30,30))
for i in range(1,10):
    plt.subplot(3,3,i)
    X = np.abs(librosa.stft(plot_audios[i][0], n_fft = n_fft, hop_length = hop_length))
    plt.plot(X)
    plt.xlabel("Frequency")
    plt.ylabel("Amplitude");
    try:
        plt.title("Fourier Transform of the sound of " + decoder[int(plot_files[i][-6:-4])] )
    except:
        plt.title("Fourier Transform of the sound of " + decoder[int(plot_files[i][-5:-4])] )

## 2.1 Spectogram

- graphical representation of the frequency content of a signal over time
    - x-axis: time
    - y-axis: frequency
    - color: the amplitude of the frequency component at that point in time

In [ ]:
plt.figure(figsize=(30,30))
for i in range(1,10):
    plt.subplot(3,3,i)
    X = librosa.stft(plot_audios[i][0])
    Xdb = librosa.amplitude_to_db(abs(X))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar()
    try:
        plt.title("Histogram of the sound of " + decoder[int(plot_files[i][-6:-4])] )
    except:
        plt.title("Histogram of the sound of " + decoder[int(plot_files[i][-5:-4])] )

Spectogram shows the frequency content of the audio signal over time:
- Temporal information: duration of or between
- Characteristics of a sound: spectral envelope or the harmonic structure
- Segment an audio file into different sound events

### 2.2 Mel Spectogram

- logarithmically renders frequencies above a certain threshold (the corner frequency)
- non-linear scale - with finer resolution at lower frequencies and coarser resolution at higher frequencies, reflecting human auditory sensitivity
- reduces the dimensionality of the frequency due to mel bins that summarize nearby frequencies 

In [ ]:
plt.figure(figsize=(30,30))
for i in range(1,10):
    plt.subplot(3,3,i)
    X, _ = librosa.effects.trim(plot_audios[i][0])
    XS = librosa.feature.melspectrogram(y=X, sr=sr)
    Xdb = librosa.amplitude_to_db(XS, ref=np.max)
    librosa.display.specshow(Xdb, sr=plot_audios[i][1], x_axis='time', y_axis='hz')
    plt.colorbar()
    try:
        plt.title("Histogram of the sound of " + decoder[int(plot_files[i][-6:-4])] )
    except:
        plt.title("Histogram of the sound of " + decoder[int(plot_files[i][-5:-4])] )